In [6]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# =========================
# Settings
# =========================

n_estimators = 300
contamination = "auto"
random_state = 42

train_fraction = 0.70

threshold_list = [0.0, -0.15, -0.10, -0.20]

data_path = Path("../data_processed/NAB/industrial_machine_features_labeled.csv")
results_root = Path("../results/iForest_NAB_thresholds")
results_root.mkdir(parents=True, exist_ok=True)

# =========================
# Load data
# =========================

df = pd.read_csv(data_path)

feature_cols = ["value", "diff1", "roll_mean", "roll_std", "roll_min", "roll_max"]
required_cols = feature_cols + ["y_true"]

df = df.dropna(subset=required_cols).reset_index(drop=True)

if "timestamp" in df.columns:
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp").reset_index(drop=True)

# =========================
# Chronological train/test split
# =========================

split_idx = int(len(df) * train_fraction)

train_window = df.iloc[:split_idx].copy()
test_window = df.iloc[split_idx:].copy()

X_train_normal_raw = train_window.loc[
    train_window["y_true"] == 0,
    feature_cols
].copy()

X_test_raw = test_window[feature_cols].copy()
y_test = test_window["y_true"].astype(int).copy()

if len(X_train_normal_raw) == 0:
    raise ValueError("The training window does not contain normal samples.")

print("Split type: chronological windows, no random sampling")
print("Train fraction:", train_fraction)
print("Total samples after preprocessing:", len(df))
print("Train window samples:", len(train_window))
print("Train normal samples:", len(X_train_normal_raw))
print("Test window samples:", len(test_window))
print("Test normal samples:", int((y_test == 0).sum()))
print("Test anomaly samples:", int((y_test == 1).sum()))

# =========================
# Scaling
# =========================

scaler = StandardScaler()

X_train = pd.DataFrame(
    scaler.fit_transform(X_train_normal_raw),
    columns=feature_cols,
    index=X_train_normal_raw.index
)

X_test = pd.DataFrame(
    scaler.transform(X_test_raw),
    columns=feature_cols,
    index=X_test_raw.index
)

# =========================
# Train fixed Isolation Forest
# =========================

model = IsolationForest(
    n_estimators=n_estimators,
    contamination=contamination,
    max_samples="auto",
    max_features=1.0,
    bootstrap=False,
    random_state=random_state,
    n_jobs=-1
)

model.fit(X_train)

pred_default_raw = model.predict(X_test)
decision_scores = model.decision_function(X_test).ravel()
score_samples = model.score_samples(X_test).ravel()

print("Decision score min:", decision_scores.min())
print("Decision score max:", decision_scores.max())
print("Decision score mean:", decision_scores.mean())

# =========================
# Helper
# =========================

def safe_name(value):
    return str(value).replace(".", "_").replace("-", "minus_")

# =========================
# Threshold experiments
# =========================

all_results = []

for threshold in threshold_list:
    exp_name = (
        f"train_fraction_{safe_name(train_fraction)}_"
        f"n_estimators_{n_estimators}_"
        f"contamination_{contamination}_"
        f"threshold_{safe_name(threshold)}"
    )

    exp_dir = results_root / exp_name
    exp_dir.mkdir(parents=True, exist_ok=True)

    df_temp = test_window.copy()

    df_temp["iforest_pred_default_raw"] = pred_default_raw
    df_temp["iforest_pred_default"] = (pred_default_raw == -1).astype(int)
    df_temp["iforest_decision"] = decision_scores
    df_temp["iforest_score"] = score_samples
    df_temp["threshold"] = threshold

    # 0 = normal, 1 = anomaly
    df_temp["is_anomaly"] = (df_temp["iforest_decision"] < threshold).astype(int)

    y_pred = df_temp["is_anomaly"].astype(int)

    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    accuracy = accuracy_score(y_test, y_pred) * 100
    precision = precision_score(y_test, y_pred, zero_division=0) * 100
    recall = recall_score(y_test, y_pred, zero_division=0) * 100
    f1 = f1_score(y_test, y_pred, zero_division=0) * 100

    result_row = {
        "experiment": exp_name,
        "train_fraction": train_fraction,
        "n_estimators": n_estimators,
        "contamination": contamination,
        "random_state": random_state,
        "threshold": threshold,
        "split_type": "chronological_window_split",
        "train_window_samples": len(train_window),
        "train_normal_samples": len(X_train),
        "test_window_samples": len(X_test),
        "test_normal_samples": int((y_test == 0).sum()),
        "test_anomaly_samples": int((y_test == 1).sum()),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "accuracy_%": round(accuracy, 2),
        "precision_%": round(precision, 2),
        "recall_%": round(recall, 2),
        "f1_%": round(f1, 2)
    }

    all_results.append(result_row)

    df_temp.to_csv(exp_dir / "predictions.csv", index=False)

    print("======================================")
    print("Isolation Forest NAB")
    print("======================================")
    print(f"Train fraction: {train_fraction}")
    print(f"Threshold: {threshold}")
    print(f"Accuracy : {accuracy:.2f}%")
    print(f"Precision: {precision:.2f}%")
    print(f"Recall   : {recall:.2f}%")
    print(f"F1-score : {f1:.2f}%")
    print("Confusion Matrix")
    print(cm)

    # =========================
    # Metrics table
    # =========================

    metrics_df = pd.DataFrame({
        "Metrika": [
            "Presnosť (Accuracy)",
            "Precíznosť (Precision)",
            "Citlivosť (Recall)",
            "F1-skóre"
        ],
        "Hodnota (%)": [
            round(accuracy, 2),
            round(precision, 2),
            round(recall, 2),
            round(f1, 2)
        ]
    })

    fig, ax = plt.subplots(figsize=(6, 2))
    ax.axis("off")

    table = ax.table(
        cellText=metrics_df.values,
        colLabels=metrics_df.columns,
        loc="center",
        cellLoc="center"
    )

    table.auto_set_font_size(False)
    table.set_fontsize(12)
    table.scale(1, 2)

    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_text_props(weight="bold")
            cell.set_facecolor("#dce6f1")
        else:
            cell.set_facecolor("#f9f9f9")

    plt.title(
        f"Výsledné metriky Isolation Forest – NAB\n"
        f"Train fraction = {train_fraction}, Threshold = {threshold}",
        fontsize=12,
        pad=20
    )

    plt.savefig(exp_dir / "metrics_table.png", dpi=300, bbox_inches="tight")
    plt.close()

    # =========================
    # Confusion matrix
    # =========================

    plt.figure(figsize=(6, 5))
    plt.imshow(cm, interpolation="nearest")
    plt.title(f"Matica zámen – Threshold = {threshold}")
    plt.colorbar()

    classes = ["Normálne", "Anomálie"]
    tick_marks = np.arange(len(classes))

    plt.xticks(tick_marks, classes)
    plt.yticks(tick_marks, classes)

    threshold_cm = cm.max() / 2.0 if cm.max() > 0 else 0.5

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            color = "black" if cm[i, j] > threshold_cm else "white"
            plt.text(
                j, i, cm[i, j],
                ha="center",
                va="center",
                color=color
            )

    plt.xlabel("Predikované triedy")
    plt.ylabel("Skutočné triedy")
    plt.tight_layout()
    plt.savefig(exp_dir / "confusion_matrix.png", dpi=300, bbox_inches="tight")
    plt.close()

    # =========================
    # Anomaly plot
    # =========================

    if "timestamp" in df_temp.columns:
        x_axis = pd.to_datetime(df_temp["timestamp"])
        x_label = "Čas"
    else:
        x_axis = np.arange(len(df_temp))
        x_label = "Index"

    plt.figure(figsize=(14, 5))
    plt.plot(x_axis, df_temp["value"], linewidth=1, label="Teplota")

    anomaly_idx = df_temp["is_anomaly"] == 1

    plt.scatter(
        np.array(x_axis)[anomaly_idx],
        df_temp.loc[anomaly_idx, "value"],
        s=18,
        label="Detegované anomálie"
    )

    plt.title(
        f"Detekcia anomálií pomocou Isolation Forest – NAB\n"
        f"Train fraction = {train_fraction}, Threshold = {threshold}"
    )
    plt.xlabel(x_label)
    plt.ylabel("Hodnota")
    plt.legend()
    plt.tight_layout()
    plt.savefig(exp_dir / "anomaly_plot.png", dpi=300, bbox_inches="tight")
    plt.close()

    # =========================
    # Score histogram
    # =========================

    plt.figure(figsize=(10, 5))

    plt.hist(
        decision_scores[y_test.values == 0],
        bins=50,
        alpha=0.7,
        label="Normálne"
    )

    plt.hist(
        decision_scores[y_test.values == 1],
        bins=50,
        alpha=0.7,
        label="Anomálie"
    )

    plt.axvline(
        threshold,
        linestyle="--",
        linewidth=2,
        label=f"Threshold = {threshold}"
    )

    plt.title(
        f"Histogram skóre Isolation Forest – NAB\n"
        f"Train fraction = {train_fraction}, Threshold = {threshold}"
    )
    plt.xlabel("Decision score")
    plt.ylabel("Počet vzoriek")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(exp_dir / "score_histogram.png", dpi=300, bbox_inches="tight")
    plt.close()

# =========================
# Save summary
# =========================

results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values(by="f1_%", ascending=False).reset_index(drop=True)

results_df.to_csv(results_root / "summary_results.csv", index=False)

print("Done")

Split type: chronological windows, no random sampling
Train fraction: 0.7
Total samples after preprocessing: 22646
Train window samples: 15852
Train normal samples: 14718
Test window samples: 6794
Test normal samples: 5660
Test anomaly samples: 1134
Decision score min: -0.2505944241112973
Decision score max: 0.13219413005383468
Decision score mean: -0.0001601604174893946
Isolation Forest NAB
Train fraction: 0.7
Threshold: 0.0
Accuracy : 82.81%
Precision: 49.26%
Recall   : 100.00%
F1-score : 66.01%
Confusion Matrix
[[4492 1168]
 [   0 1134]]
Isolation Forest NAB
Train fraction: 0.7
Threshold: -0.15
Accuracy : 89.09%
Precision: 66.58%
Recall   : 69.58%
F1-score : 68.05%
Confusion Matrix
[[5264  396]
 [ 345  789]]
Isolation Forest NAB
Train fraction: 0.7
Threshold: -0.1
Accuracy : 88.78%
Precision: 62.79%
Recall   : 80.51%
F1-score : 70.56%
Confusion Matrix
[[5119  541]
 [ 221  913]]
Isolation Forest NAB
Train fraction: 0.7
Threshold: -0.2
Accuracy : 89.28%
Precision: 74.40%
Recall   : 54